In [ ]:
# ============================================================
# Coupled Raman Mechanistic Model 2 (CRMM2) -- Scenarios 1 & 2
# ----------------------------------------------------------
# Second mechanistic baseline used to benchmark CRHM against: same
# Pyomo.DAE spectral curve-resolution formulation, but with Michaelis-
# Menten saturation kinetics (v = Vmax*C/(C+Km), no inhibition term)
# instead of an ANN-corrected kinetic term. Everything else (spectral
# unmixing, objective, discretization, multistart solve) mirrors CRHM.
# ============================================================

import math
import numpy as np
import pandas as pd
from datetime import datetime
import time as time_mod

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Pyomo: algebraic modeling language used to build the nonlinear program (NLP)
# that couples the spectral unmixing and kinetic constraints.
from pyomo.environ import *
# Pyomo.DAE: adds ContinuousSet/DerivativeVar and collocation discretization
# so the ODE kinetics can be embedded as algebraic constraints in the NLP.
from pyomo.dae import *

# solve_ivp is used to (a) generate ground-truth concentration trajectories
# and (b) re-simulate the fitted mechanistic model for out-of-sample testing.
from scipy.integrate import odeint, solve_ivp
from scipy.interpolate import CubicSpline
from sklearn.metrics import r2_score, mean_squared_error

# Parallelizes the out-of-sample predictive-performance sweep across many
# initial conditions (see predictive_perf_testing calls below).
from joblib import Parallel, delayed

import pickle
import random

import os
# Works around an OpenMP "duplicate runtime" crash that can occur when
# multiple libraries link their own copies of libomp.
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import gc

# pytorch relates imports
import torch
import torch.nn as nn
import torch.optim as optim

# imports from captum library
# Captum / torch imports are carried over from the CRHM notebook template;
# this purely-mechanistic baseline has no ANN, so they are unused here.
from captum.attr import IntegratedGradients, Saliency, Occlusion

In [ ]:
# Ground-truth data-generating process: two enzyme-catalyzed reactions
# A -> B -> C, each following Michaelis-Menten kinetics with noncompetitive
# product inhibition by C. Used both to synthesize training data and as the
# reference trajectory when scoring predictive performance of fitted models.
# Note this mechanistic baseline's *fitted* rate law (below) does NOT include
# this inhibition term -- that mismatch is exactly what CRHM's ANN correction
# is meant to compensate for.
def case_study1(t, y, Ki):
    Ca, Cb, Cc = y
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    v1max = 4.5
    v2max = 2.5
    
    Km1 = 5
    Km2 = 5
    
    Ki1 = Ki[0]
    Ki2 = Ki[1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    inh1 = (1 + (Cc/Ki1))
    inh2 = (1 + (Cc/Ki2))
    
    # Noncompetitive Inhibition    
    v1 = (v1max*Ca)/((Ca + Km1)*inh1)
    v2 = (v2max*Cb)/((Cb + Km2)*inh2)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    dCadt = -v1
    dCbdt = v1 - v2
    dCcdt = v2
    
    # Order matches the y = [Ca, Cb, Cc] state vector passed in by solve_ivp.
    return [dCadt, dCbdt, dCcdt]

# Coupled Raman Phenom Methodology

In [ ]:
# Pyomo.DAE constraint rules for dCa/dCb/dCc per (dataset d, time t), using
# v = Vmax*C/(C+Km) Michaelis-Menten saturation kinetics (no inhibition term)
# for both reactions -- this is the deliberately-simplified rate law that
# distinguishes this mechanistic baseline from CRHM's ANN-corrected version.
# At t=0 each returns an equality to the known initial concentration;
# otherwise it returns the ODE residual.
def dCadt_int_con(model, d, t):
    if t == 0:
        return model.conc_Ca[d, t] == model.conc_Ca_initial[d]
    
    v1 = (model.v1m*model.conc_Ca[d,t])/(model.conc_Ca[d, t] + model.Km1)
    
    return model.dCadt[d, t] == -1*v1


def dCbdt_int_con(model, d, t):
    if t == 0:
        return model.conc_Cb[d, t] == model.conc_Cb_initial[d]
    
    v1 = (model.v1m*model.conc_Ca[d,t])/(model.conc_Ca[d, t] + model.Km1)
    v2 = (model.v2m*model.conc_Cb[d,t])/(model.conc_Cb[d, t] + model.Km2)
    
    return model.dCbdt[d, t] == v1 - v2


def dCcdt_int_con(model, d, t):
    if t == 0:
        return model.conc_Cc[d, t] == model.conc_Cc_initial[d]
    
    v2 = (model.v2m*model.conc_Cb[d,t])/(model.conc_Cb[d, t] + model.Km2)
    
    return model.dCcdt[d, t] == v2

In [ ]:
# Beer-Lambert-type constraints: each species' contribution to the measured
# spectrum at (dataset d, time t, wavenumber wv) is its pure spectrum times
# its concentration. sp_con_* are auxiliary variables tied to conc_* via
# these equalities so the objective can compare against summed spectra.
def sp_c_Ca(model, d, t, wv):
    return model.sp_con_Ca[d, t, wv] == model.pure_Sa[wv]*model.conc_Ca[d, t]

def sp_c_Cb(model, d, t, wv):
    return model.sp_con_Cb[d, t, wv] == model.pure_Sb[wv]*model.conc_Cb[d, t]

def sp_c_Cc(model, d, t, wv):
    return model.sp_con_Cc[d, t, wv] == model.pure_Sc[wv]*model.conc_Cc[d, t]

In [ ]:
# Curve-resolution objective: mean squared error between the measured
# spectral data D and the additive (Beer-Lambert) reconstruction from all
# three species' spectral contributions, averaged over all
# datasets/time points/wavenumbers.
def D_mse_objective(model):
    fin = 0
    
    for i in model.d_idx:
        for j in model.t_meas:
            for k in model.wv_meas:
                fin += (model.D_final_meas[i,j,k] - model.sp_con_Ca[i,j,k] 
                        - model.sp_con_Cb[i,j,k] - model.sp_con_Cc[i,j,k])**2
                
    final = fin/(len(model.d_idx)*len(model.t_meas)*len(model.wv_meas))
    return final

In [ ]:
# Builds and solves the Pyomo NLP for one Scenarios 1 & 2 training condition
# using the CRMM2 mechanistic rate law (no ANN). Inputs: t_fin/wv_fin
# (measurement grids), D_full (stacked experiment x time x wavenumber
# spectral data), C0 (initial concentrations per experiment).
def coupled_Raman_formulation(t_fin, wv_fin, D_full, C0):
    
    nd = len(C0)
    nt = len(t_fin)
    nw = len(wv_fin)

    # D_full is stacked as (experiment*time) rows x wavenumber columns; unpack it
    # into a {(dataset, time, wavenumber): value} dict for Pyomo Param.initialize.
    D_dict = {}

    for i in range(nd):
        for j in range(nt):
            for k in range(nw):
                l = i*(nt) + j
                D_dict[(i, t_fin[j], wv_fin[k])] = D_full[l, k]

    # Per-experiment initial concentrations, keyed by dataset index for Pyomo Param.
    Ca_initial = {}
    Cb_initial = {}
    Cc_initial = {}

    for i in range(nd):
        Ca_initial[i] = C0[i,0]
        Cb_initial[i] = C0[i,1]
        Cc_initial[i] = C0[i,2]
    
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ##~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    # ---- Pyomo model construction begins here ----
    model = ConcreteModel()

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Data Related Sets - experiment number, time point and wavenumber

    model.d_idx = Set(initialize = range(nd))
    model.t_meas = Set(initialize = t_fin)
    model.wv_meas = Set(initialize = wv_fin)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Parameters for this formulation - Measured Concentration and Known Etot conc

    # Measured spectral data and known initial concentrations enter as fixed Params.
    model.D_final_meas = Param(model.d_idx, model.t_meas, model.wv_meas, initialize = D_dict)

    model.conc_Ca_initial = Param(model.d_idx, initialize = Ca_initial)
    model.conc_Cb_initial = Param(model.d_idx, initialize = Cb_initial)
    model.conc_Cc_initial = Param(model.d_idx, initialize = Cc_initial)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Variables for species and their derivative variables for each respectibe species

    # Decision variables: unknown pure-component spectra (nonnegative, bounded),
    # their per-(dataset,time,wavenumber) spectral contributions, the continuous
    # time set for the DAE, concentration trajectories, and their time derivatives.
    model.pure_Sa = Var(model.wv_meas, within = NonNegativeReals, bounds = (0, 100))
    model.pure_Sb = Var(model.wv_meas, within = NonNegativeReals, bounds = (0, 100))
    model.pure_Sc = Var(model.wv_meas, within = NonNegativeReals, bounds = (0, 100))

    model.sp_con_Ca = Var(model.d_idx, model.t_meas, model.wv_meas, within = NonNegativeReals)
    model.sp_con_Cb = Var(model.d_idx, model.t_meas, model.wv_meas, within = NonNegativeReals)
    model.sp_con_Cc = Var(model.d_idx, model.t_meas, model.wv_meas, within = NonNegativeReals)

    model.time = ContinuousSet(initialize=model.t_meas, bounds=(t_fin[0], t_fin[-1]))

    model.conc_Ca = Var(model.d_idx, model.time, within = NonNegativeReals)
    model.conc_Cb = Var(model.d_idx, model.time, within = NonNegativeReals)
    model.conc_Cc = Var(model.d_idx, model.time, within = NonNegativeReals)

    model.dCadt = DerivativeVar(model.conc_Ca, within = Reals)
    model.dCbdt = DerivativeVar(model.conc_Cb, within = Reals)
    model.dCcdt = DerivativeVar(model.conc_Cc, within = Reals)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Variables for kinetic Parameters within the ODE kinetic system

    # Kinetic parameters: Vmax-like rate constants (v1m/v2m) and Michaelis
    # constants (Km1/Km2), all fit as free variables -- but note there is still
    # no inhibition term, unlike the ground-truth model.
    model.v1m = Var(within = NonNegativeReals, bounds = (0, 10))
    model.v2m = Var(within = NonNegativeReals, bounds = (0, 10))
    
    model.Km1 = Var(within = NonNegativeReals, bounds = (0, 10))
    model.Km2 = Var(within = NonNegativeReals, bounds = (0, 10))

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Spectral Constraints

    # Wire up the Beer-Lambert spectral constraints for all three species.
    model.spectral_Ca = Constraint(model.d_idx, model.t_meas, model.wv_meas, rule = sp_c_Ca)
    model.spectral_Cb = Constraint(model.d_idx, model.t_meas, model.wv_meas, rule = sp_c_Cb)
    model.spectral_Cc = Constraint(model.d_idx, model.t_meas, model.wv_meas, rule = sp_c_Cc)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Mechanistic Constraints

    # Wire up the (non-ANN) mechanistic DAE kinetic constraints (and t=0 initial
    # conditions).
    model.deriv_Ca = Constraint(model.d_idx, model.time, rule = dCadt_int_con)
    model.deriv_Cb = Constraint(model.d_idx, model.time, rule = dCbdt_int_con)
    model.deriv_Cc = Constraint(model.d_idx, model.time, rule = dCcdt_int_con)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Objective function

    # Single objective: minimize spectral reconstruction MSE across all constraints.
    model.obj = Objective(expr = D_mse_objective)

    # # Discretizer for Pyomo.DAE (Differential-Algebraic Equations)
    # Convert the continuous-time DAE into a finite set of algebraic collocation
    # equations (orthogonal collocation on finite elements) so IPOPT can solve it
    # as a standard NLP. nfe = number of finite elements, ncp = collocation points
    # per element.
    discretizer = TransformationFactory('dae.collocation')
    discretizer.apply_to(model, nfe=len(t_fin), ncp=2, scheme='LAGRANGE-RADAU')  # Apply Lagrange-Radau collocation method

    # Multistart NLP solve: IPOPT is run from many random initializations with a
    # per-restart wall-time cap, and the best feasible result is kept.
    solver = SolverFactory("multistart")

    results = solver.solve(
        model,
        suppress_unbounded_warning=True,
        iterations=50,
        solver="ipopt",
        solver_args={"options": {"max_cpu_time": 30.0, 
                                 "max_iter": 3000}})

    # Print solver status and results
    print("Solver status:", results.solver.status)
    print("Termination Condition:", results.solver.termination_condition)
    print("Objective: ", value(model.obj))  # Print the objective function value
    
    return model

# Hybrid Model and Recovered Spectra Testing

In [ ]:
# Pulls the solved Pyomo Var values (fitted Vmax-like rate constants v1m/v2m
# and Michaelis constants Km1/Km2) out into a plain dict, so the fitted
# mechanistic model can be re-simulated quickly outside of Pyomo (via scipy
# solve_ivp instead of re-solving an NLP each time).
def TIV_param(model):
    
    K_params = {}

    K_params["v1m"] = value(model.v1m)
    K_params["v2m"] = value(model.v2m)
    K_params["K1m"] = value(model.Km1)
    K_params["K2m"] = value(model.Km2)
    
    return [K_params]

In [ ]:
# Phenomenological/mechanistic model integration (PMI) ODE system: same
# A -> B -> C structure as case_study1, but using the fitted Michaelis-Menten
# saturation rate law r = Vmax*c/(c+Km) (no inhibition term) instead of the
# ground truth's Michaelis-Menten + noncompetitive inhibition kinetics.
def PMI_CS1_system(t, y, K_params):
    
    c1, c2, c3 = y
        
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    v1m = K_params["v1m"]
    v2m = K_params["v2m"]
    
    k1m = K_params["K1m"]
    k2m = K_params["K2m"]
    
    r1 = (v1m*c1)/(c1 + k1m)
    r2 = (v2m*c2)/(c2 + k2m)
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~    

    dc1dt = -1*r1
    
    dc2dt = r1 - r2
    
    dc3dt = r2
        
    return [dc1dt, dc2dt, dc3dt]

In [ ]:
# Simulates ground truth vs. fitted mechanistic model from a new initial
# condition (not used during fitting) and scores agreement per species via
# R^2, RMSE, and NRMSE. Negative R^2 (worse than predicting the mean) is
# clipped to 0 for reporting.
def predictive_perf_testing(ic, tspan, Ki, K_params):
    
    r2_list = []
    rmse_list = []
    nrmse_list = []

    sol_true = solve_ivp(case_study1, 
                    tspan, 
                    ic,
                    args = Ki,
                    method = "Radau")
    
    teval = sol_true.t

    sol_calc = solve_ivp(PMI_CS1_system, 
                        tspan, 
                        ic, 
                        t_eval = teval, 
                        args = tuple(K_params), 
                        method = "Radau")

    y_true = sol_true.y
    y_calc = sol_calc.y

    for j in range(len(y_true)):
        mse = mean_squared_error(y_true[j], y_calc[j])
        rmse_list.append(mse**0.5)
        nrmse_list.append((mse**0.5)/np.mean(y_true[j]))
        
        if r2_score(y_true[j], y_calc[j]) < 0:
            r2_list.append(0)
        else:
            r2_list.append(r2_score(y_true[j], y_calc[j]))
        
    return tuple([np.mean(r2_list), np.mean(rmse_list), np.mean(nrmse_list)])

In [ ]:
# Compares the fitted pure-component spectra against the known ground-truth
# spectra: cosine similarity (sm) between L2-normalized spectra, plus NRMSE
# of the raw (unnormalized) difference, one entry per species/component.
def spectra_recovery_metrics(ST_ground, model):
    
    groundtruth_spectra = ST_ground
    
    ps_list = [model.pure_Sa, model.pure_Sb, model.pure_Sc]
    recovered_spectra = []
    for i in range(len(ps_list)):
        spc = []
        for j in model.wv_meas:
            spc.append(value(ps_list[i][value(j)]))
            
        spc = np.array(spc)
        recovered_spectra.append(spc)
    recovered_spectra = np.array(recovered_spectra)
    
    spectral_metric = {}
    
    for i in range(len(groundtruth_spectra)):    
        gs = groundtruth_spectra[i,:]
        gs_n = gs / np.linalg.norm(gs)

        rs = recovered_spectra[i,:] 
        rs_n = rs / np.linalg.norm(rs)

        err = gs - rs

        sm = np.matmul(gs_n, rs_n.T)
        nrmse = np.linalg.norm(err)/np.linalg.norm(gs)
        
        ky = "Component " + str(i+1)
        spectral_metric[ky] = [sm, nrmse]
    
    return spectral_metric

In [ ]:
# Defines the out-of-sample test sweep: 50 log-spaced initial concentrations
# of Ca (0.01-10 mM) each paired with Cb0 = Cc0 = 0, used to probe predictive
# performance across a wide range of substrate loading.
C1_range = np.logspace(-2, 1, 50)

rect_init = []

for i in range(len(C1_range)):
    rect_init.append(np.array([C1_range[i], 0, 0]))

In [ ]:
# Scenario 1 run: iterates over every training condition file, fits
# CRMM2, scores predictive + spectral recovery performance, and
# pickles the full result dictionary (including compute time) to
# "Coupled Performance/Mechanistic model - v2/Scenario 1/".
train_scenario1 = os.listdir("Train Conditions/Scenario 1")
for i in range(len(train_scenario1)):

    file = os.path.join("Train Conditions/Scenario 1", train_scenario1[i])

    with open(file, "rb") as f:
        full_data = pickle.load(f)

    t_fin = full_data["t_fin"]
    wv_fin = full_data["wv_fin"].reshape(-1,)
    D_full = full_data["D_full"]
    C0 = full_data["C0"]
    ST_ground = full_data["ST_ground"]

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("\033[1mTime start of Opt: \033[0m", datetime.now())

    start_compute = time_mod.time()
    
    model = coupled_Raman_formulation(t_fin, wv_fin, D_full, C0)
        
    end_compute = time_mod.time()

    print("\033[1mTime end of Opt: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    compute_time = end_compute - start_compute
    
    K_params = TIV_param(model)
    
    rs_ST = int(train_scenario1[i].split("_")[3])
    rs_D = float(train_scenario1[i].split("_")[-1].split(".")[0])

    Ki1 = float(train_scenario1[i].split("_")[1])
    Ki2 = float(train_scenario1[i].split("_")[2])
    Ki = tuple([[Ki1, Ki2]])
    spc_ovp = float(train_scenario1[i].split("_")[4])

    test_input = []
    for m in range(len(rect_init)):
        test_input.append(tuple([rect_init[m], (0, 20), Ki, K_params]))

    conc_predictive_performance = Parallel(n_jobs=-1)(
        delayed(predictive_perf_testing)(ic, t, k, KP) for ic, t, k, KP in test_input)

    spectra_predictive_performance = spectra_recovery_metrics(ST_ground, model)

    final_performance_dict = {}
    final_performance_dict["conc_perf"] = conc_predictive_performance
    final_performance_dict["spectra_perf"] = spectra_predictive_performance
    final_performance_dict["K params"] = [K_params]
    final_performance_dict["Compute Time"] = compute_time

    print("\033[1mTime end of Test: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    path_name = "Coupled Performance/Mechanistic model - v2/Scenario 1/"
    file_name = "Perf_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(final_performance_dict, f)

    del model
    gc.collect()

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

In [ ]:
# Scenario 2 run: iterates over every training condition file, fits
# CRMM2, scores predictive + spectral recovery performance, and
# pickles the full result dictionary (including compute time) to
# "Coupled Performance/Mechanistic model - v2/Scenario 2/".
train_scenario2 = os.listdir("Train Conditions/Scenario 2")
for i in range(len(train_scenario2)):

    file = os.path.join("Train Conditions/Scenario 2", train_scenario2[i])

    with open(file, "rb") as f:
        full_data = pickle.load(f)

    t_fin = full_data["t_fin"]
    wv_fin = full_data["wv_fin"].reshape(-1,)
    D_full = full_data["D_full"]
    C0 = full_data["C0"]
    ST_ground = full_data["ST_ground"]

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("\033[1mTime start of Opt: \033[0m", datetime.now())

    start_compute = time_mod.time()
    
    model = coupled_Raman_formulation(t_fin, wv_fin, D_full, C0)
        
    end_compute = time_mod.time()

    print("\033[1mTime end of Opt: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    compute_time = end_compute - start_compute
    
    K_params = TIV_param(model)
    
    rs_ST = int(train_scenario2[i].split("_")[3])
    rs_D = float(train_scenario2[i].split("_")[-1].split(".")[0])

    Ki1 = float(train_scenario2[i].split("_")[1])
    Ki2 = float(train_scenario2[i].split("_")[2])
    Ki = tuple([[Ki1, Ki2]])
    spc_ovp = float(train_scenario2[i].split("_")[4])

    test_input = []
    for m in range(len(rect_init)):
        test_input.append(tuple([rect_init[m], (0, 20), Ki, K_params]))

    conc_predictive_performance = Parallel(n_jobs=-1)(
        delayed(predictive_perf_testing)(ic, t, k, KP) for ic, t, k, KP in test_input)

    spectra_predictive_performance = spectra_recovery_metrics(ST_ground, model)

    final_performance_dict = {}
    final_performance_dict["conc_perf"] = conc_predictive_performance
    final_performance_dict["spectra_perf"] = spectra_predictive_performance
    final_performance_dict["K params"] = [K_params]
    final_performance_dict["Compute Time"] = compute_time

    print("\033[1mTime end of Test: \033[0m", datetime.now())

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    path_name = "Coupled Performance/Mechanistic model - v2/Scenario 2/"
    file_name = "Perf_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(final_performance_dict, f)

    del model
    gc.collect()

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")